In [ ]:
'''!pip install -q datasets'''

In [ ]:
'''!cp -r /kaggle/input/datasets/suryaghosh007/cifar-100 /kaggle/working/
!cp -r /kaggle/input/models/suryaghosh007/resnet-32-latest/pytorch/default/1 /kaggle/working/'''

In [ ]:
!cp -r /kaggle/input/datasets/pankrzysiu/cifar10-python /kaggle/working/

In [1]:
!cp -r /kaggle/input/datasets/chanplusplus/imagenetlt /kaggle/working/

^C


In [ ]:
# Cell 1: Setup, Dataset Loading & Analysis
import os, random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, top_k_accuracy_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 1000  # ImageNet has 1000 classes
BATCH_SIZE = 128
SEED = 42

# Point DATA_ROOT to the folder containing the txt files and torch_image_folder[cite: 6]
DATA_ROOT = "/kaggle/input/imagenet-lt/ImageNet-LT" 
OUT_DIR = "./outputs"
os.makedirs(OUT_DIR, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
print(f"Device: {DEVICE}")

# Standard ImageNet Mean and STD
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

train_tf = T.Compose([
    T.RandomResizedCrop(224),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

test_tf = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(MEAN, STD)
])

# Custom Dataset to read ImageNet-LT txt files and load from torch_image_folder[cite: 6]
class ImageNetLTDataset(Dataset):
    def __init__(self, root_dir, txt_file, transform=None):
        self.img_folder = os.path.join(root_dir, "torch_image_folder")
        self.transform = transform
        self.data = []
        self.targets = []
        
        with open(os.path.join(root_dir, txt_file), 'r') as f:
            for line in f:
                img_path, label = line.strip().split()
                self.data.append(img_path)
                self.targets.append(int(label))
                
    def __len__(self): 
        return len(self.targets)
        
    def __getitem__(self, i):
        path = os.path.join(self.img_folder, self.data[i])
        img = Image.open(path).convert('RGB')
        
        if self.transform:
            img = self.transform(img)
            
        return {
            "pixel_values": img,
            "fine_label": self.targets[i]
        }

# Load Datasets[cite: 6]
train_ds = ImageNetLTDataset(DATA_ROOT, "ImageNet_LT_train.txt", train_tf)
test_ds  = ImageNetLTDataset(DATA_ROOT, "ImageNet_LT_test.txt", test_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

# Calculate class frequencies directly from the pre-imbalanced training set
CLS_COUNTS = np.bincount(train_ds.targets, minlength=NUM_CLASSES)
MANY_CLASSES   = np.where(CLS_COUNTS > 100)[0]
MEDIUM_CLASSES = np.where((CLS_COUNTS <= 100) & (CLS_COUNTS >= 20))[0]
FEW_CLASSES    = np.where(CLS_COUNTS < 20)[0]

print(f"LT train size: {len(train_ds.targets)}")
print(f"many: {len(MANY_CLASSES)}  medium: {len(MEDIUM_CLASSES)}  few: {len(FEW_CLASSES)}")


# Cell 2: Universal Evaluation Function (With NaN safety)
@torch.no_grad()
def evaluate(model, loader, is_deit=False):
    model.eval()
    all_logits, all_labels = [], []
    total_loss = 0.0
    
    for batch in loader:
        x = batch["pixel_values"].to(DEVICE)
        y = batch["fine_label"].to(DEVICE)
        
        logits = model(x)
        
        loss = F.cross_entropy(logits, y)
        total_loss += loss.item() * x.size(0)
        
        all_logits.append(logits.cpu())
        all_labels.append(y.cpu())
        
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    preds = logits.argmax(1)

    avg_loss = total_loss / len(labels)
    top1 = (preds == labels).mean() * 100
    top5 = top_k_accuracy_score(labels, logits, k=5, labels=list(range(NUM_CLASSES))) * 100
    
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")
    precision = precision_score(labels, preds, average="macro", zero_division=0)
    recall = recall_score(labels, preds, average="macro", zero_division=0)
    cm = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))

    per_class_acc = np.zeros(NUM_CLASSES)
    for c in range(NUM_CLASSES):
        mask = labels == c
        per_class_acc[c] = (preds[mask] == labels[mask]).mean() * 100 if mask.sum() > 0 else np.nan

    # Safe evaluation that prevents NaN warnings if a category has 0 classes
    group_acc = {
        "many": float(np.nanmean(per_class_acc[MANY_CLASSES])) if len(MANY_CLASSES) > 0 else 0.0,
        "medium": float(np.nanmean(per_class_acc[MEDIUM_CLASSES])) if len(MEDIUM_CLASSES) > 0 else 0.0,
        "few": float(np.nanmean(per_class_acc[FEW_CLASSES])) if len(FEW_CLASSES) > 0 else 0.0,
    }
    
    return {
        "loss": avg_loss,
        "top1": top1, 
        "top5": top5, 
        "macro_f1": macro_f1, 
        "weighted_f1": weighted_f1,
        "precision_macro": precision,
        "recall_macro": recall,
        "group_acc": group_acc,
        "per_class_acc": per_class_acc,
        "confusion_matrix": cm
    }

In [ ]:
# Cell 3 & 4: Training and Evaluating ResNet-32 Teacher Model 
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# 1. Define Custom ResNet-32 Architecture for CIFAR
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride, bias=False),
                nn.BatchNorm2d(planes))
                
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class ResNetCIFAR(nn.Module):
    # ResNet-32 = 3 stages x 5 blocks, He et al. CIFAR variant
    def __init__(self, num_blocks=(5,5,5), num_classes=NUM_CLASSES):
        super().__init__()
        self.in_planes = 16
        self.conv1 = nn.Conv2d(3, 16, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.layer1 = self._make_layer(16, num_blocks[0], 1)
        self.layer2 = self._make_layer(32, num_blocks[1], 2)
        self.layer3 = self._make_layer(64, num_blocks[2], 2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, num_classes)
        
    def _make_layer(self, planes, n, stride):
        strides = [stride] + [1]*(n-1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)
        
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out); out = self.layer2(out); out = self.layer3(out)
        out = self.avgpool(out).flatten(1)
        return self.fc(out)

def resnet32(num_classes=NUM_CLASSES):
    return ResNetCIFAR((5,5,5), num_classes)


# 2. Define SAM Optimizer Wrapper
class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, adaptive=False, **kwargs):
        defaults = dict(rho=rho, adaptive=adaptive, **kwargs)
        super(SAM, self).__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None: continue
                self.state[p]["old_p"] = p.data.clone()
                e_w = (torch.pow(p, 2) if group["adaptive"] else 1.0) * p.grad * scale.to(p)
                p.add_(e_w)
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                p.data = self.state[p]["old_p"]
        self.base_optimizer.step()
        if zero_grad: self.zero_grad()

    def _grad_norm(self):
        shared_device = self.param_groups[0]["params"][0].device
        norm = torch.norm(
            torch.stack([
                ((torch.abs(p) if group["adaptive"] else 1.0) * p.grad).norm(p=2).to(shared_device)
                for group in self.param_groups for p in group["params"]
                if p.grad is not None
            ]), p=2)
        return norm


# 3. Instantiate and Train the Model
resnet_teacher = resnet32(num_classes=NUM_CLASSES).to(DEVICE)
teacher_save_path = os.path.join(OUT_DIR, "resnet32_best.pt")

def train_resnet_with_sam(model, epochs=200, lr=0.1):
    base_optimizer = optim.SGD
    opt = SAM(model.parameters(), base_optimizer, rho=0.05, lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.MultiStepLR(opt.base_optimizer, milestones=[120, 160], gamma=0.1)
    criterion = nn.CrossEntropyLoss()
    
    best_val_acc = 0.0
    print("Starting Custom ResNet-32 Teacher Training with SAM...")
    
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            x, y = batch["pixel_values"].to(DEVICE), batch["fine_label"].to(DEVICE)
            
            # SAM Step 1
            loss = criterion(model(x), y)
            loss.backward()
            opt.first_step(zero_grad=True)
            
            # SAM Step 2
            criterion(model(x), y).backward()
            opt.second_step(zero_grad=True)
            
        sched.step()
        
        val = evaluate(model, test_loader)
        if val["top1"] > best_val_acc:
            best_val_acc = val["top1"]
            torch.save(model.state_dict(), teacher_save_path)
            
        if (epoch+1) % 10 == 0:
            print(f"[ResNet32] Epoch {epoch+1}/{epochs} - Val Top1: {val['top1']:.2f}% | Best: {best_val_acc:.2f}%")

train_resnet_with_sam(resnet_teacher)

# 4. Evaluate the Best Saved Model
resnet_teacher.load_state_dict(torch.load(teacher_save_path, map_location=DEVICE, weights_only=True))
resnet_eval = evaluate(resnet_teacher, test_loader)

print("\nEvaluation Results for Best ResNet-32 Teacher:")
print(f"{'Model':<14} | {'top1':<6} | {'top5':<6} | {'macro-F1':<8} | {'many':<6} | {'medium':<6} | {'few':<6}")
print("-" * 65)
print(f"{'ResNet-32':<14} | {resnet_eval['top1']:>5.1f}% | {resnet_eval['top5']:>5.1f}% | {resnet_eval['macro_f1']:>8.3f} | {resnet_eval['group_acc']['many']:>5.1f}% | {resnet_eval['group_acc']['medium']:>5.1f}% | {resnet_eval['group_acc']['few']:>4.1f}%")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os

# 1. Define Custom ResNet-32 Architecture
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride, bias=False),
                nn.BatchNorm2d(planes))
                
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class ResNetCIFAR(nn.Module):
    def __init__(self, num_blocks=(5,5,5), num_classes=10):
        super().__init__()
        self.in_planes = 16
        self.conv1 = nn.Conv2d(3, 16, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.layer1 = self._make_layer(16, num_blocks[0], 1)
        self.layer2 = self._make_layer(32, num_blocks[1], 2)
        self.layer3 = self._make_layer(64, num_blocks[2], 2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, num_classes)
        
    def _make_layer(self, planes, n, stride):
        strides = [stride] + [1]*(n-1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)
        
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out); out = self.layer2(out); out = self.layer3(out)
        out = self.avgpool(out).flatten(1)
        return self.fc(out)

def resnet32(num_classes=10):
    return ResNetCIFAR((5,5,5), num_classes)

# 2. Initialize Model & Load Saved Weights
resnet_teacher = resnet32(num_classes=NUM_CLASSES).to(DEVICE)
teacher_save_path = "./outputs/resnet32_best.pt" 

resnet_teacher.load_state_dict(torch.load(teacher_save_path, map_location=DEVICE, weights_only=True))
resnet_teacher.eval()
print(f"Successfully loaded ResNet-32 weights from {teacher_save_path}")

# 3. Run Evaluation on Test Loader
resnet_eval = evaluate(resnet_teacher, test_loader)

print("\nEvaluation Results for Loaded ResNet-32 Teacher:")
print(f"{'Model':<14} | {'top1':<6} | {'top5':<6} | {'macro-F1':<8} | {'many':<6} | {'medium':<6} | {'few':<6}")
print("-" * 65)
print(f"{'ResNet-32':<14} | {resnet_eval['top1']:>5.1f}% | {resnet_eval['top5']:>5.1f}% | {resnet_eval['macro_f1']:>8.3f} | {resnet_eval['group_acc']['many']:>5.1f}% | {resnet_eval['group_acc']['medium']:>5.1f}% | {resnet_eval['group_acc']['few']:>4.1f}%")

In [ ]:
# Cell 5 & 6: DeiT Training and Evaluation (OPTIMIZED FOR SPEED)
from timm.scheduler.cosine_lr import CosineLRScheduler
from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy
import torch.optim as optim
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import time

# 1. Provide PatchEmbed
class PatchEmbed(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_ch=3, embed_dim=192):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch_size, stride=patch_size)
        
    def forward(self, x):
        x = self.proj(x)               
        return x.flatten(2).transpose(1, 2)  

# 2. Adapt the ViT-Tiny into a DeiT-Tiny for Dual-Head Distillation
class DeiTTinyStudent(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_ch=3, num_classes=NUM_CLASSES,
                 embed_dim=192, depth=8, n_heads=3, mlp_ratio=4.0, drop=0.1):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_ch, embed_dim)
        n_patches = self.patch_embed.n_patches
        
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.dist_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 2, embed_dim))
        self.pos_drop = nn.Dropout(drop)
        
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads, dim_feedforward=int(embed_dim*mlp_ratio),
            dropout=drop, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        
        self.head_expert = nn.Linear(embed_dim, num_classes)
        self.tail_expert = nn.Linear(embed_dim, num_classes)
        
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.dist_token, std=0.02)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        
        cls = self.cls_token.expand(B, -1, -1)
        dist = self.dist_token.expand(B, -1, -1)
        
        x = torch.cat([cls, dist, x], dim=1) + self.pos_embed
        x = self.pos_drop(x)
        x = self.encoder(x)
        x = self.norm(x)
        
        fc = self.head_expert(x[:, 0]) 
        fd = self.tail_expert(x[:, 1]) 
        
        if self.training:
            return fc, fd
        return (fc + fd) / 2.0


# 3. Setup Models
resnet_teacher = resnet_teacher.to(DEVICE) 
resnet_teacher.eval() 
for param in resnet_teacher.parameters():
    param.requires_grad = False 

deit_student = DeiTTinyStudent(num_classes=NUM_CLASSES).to(DEVICE)

# 4. Training Hyperparameters 
deit_epochs = 1200
warmup_epochs = 5
drw_epoch = 1100
effective_batch_size = 512 
grad_accum_steps = max(1, effective_batch_size // BATCH_SIZE) 

opt_deit = optim.AdamW(deit_student.parameters(), lr=5e-4, weight_decay=0.05)
sched_deit = CosineLRScheduler(opt_deit, t_initial=deit_epochs, lr_min=1e-6, warmup_t=warmup_epochs, warmup_lr_init=1e-6)

criterion_ce = nn.CrossEntropyLoss(label_smoothing=0.1)
criterion_soft = SoftTargetCrossEntropy()
mixup_fn = Mixup(mixup_alpha=0.8, cutmix_alpha=1.0, label_smoothing=0.1, num_classes=NUM_CLASSES)

def drw_loss(student_logits, teacher_logits, temperature=3.0):
    loss_kl = nn.KLDivLoss(reduction="batchmean")(
        F.log_softmax(student_logits / temperature, dim=1),
        F.softmax(teacher_logits / temperature, dim=1)
    )
    return loss_kl * (temperature ** 2)

deit_save_path = os.path.join(OUT_DIR, "deit_tiny_best.pt")
best_deit_acc = 0.0

# 5. Main DeiT Training Loop
print("Starting DeiT Student Training...")
for epoch in range(deit_epochs):
    epoch_start_time = time.time()
    deit_student.train()
    is_drw = epoch >= drw_epoch
    opt_deit.zero_grad()
    
    for i, batch in enumerate(train_loader):
        x, y = batch["pixel_values"].to(DEVICE), batch["fine_label"].to(DEVICE)
        
        if not is_drw:
            x, targets = mixup_fn(x, y)
            
        fc, fd = deit_student(x)
        
        with torch.no_grad():
            teacher_logits = resnet_teacher(x)
            
        if not is_drw:
            loss_ce = criterion_soft(fc, targets)
        else:
            loss_ce = criterion_ce(fc, y) 
            
        loss_tail = drw_loss(fd, teacher_logits) if is_drw else 0.0
        
        loss = (loss_ce + loss_tail) / grad_accum_steps
        loss.backward()
        
        if (i + 1) % grad_accum_steps == 0 or (i + 1) == len(train_loader):
            opt_deit.step()
            opt_deit.zero_grad()
            
    sched_deit.step(epoch + 1) 
    epoch_duration = time.time() - epoch_start_time
    
    # Evaluate every 10 epochs initially, and EVERY epoch during the DRW phase
    if (epoch + 1) % 10 == 0 or is_drw or epoch == deit_epochs - 1:
        val = evaluate(deit_student, test_loader, is_deit=True)
        
        if val["top1"] > best_deit_acc:
            best_deit_acc = val["top1"]
            torch.save(deit_student.state_dict(), deit_save_path)
            print(f"*** New Best Model Saved at Epoch {epoch+1} with Top-1: {best_deit_acc:.2f}% ***")
            
        print(f"[DeiT EVAL] Epoch {epoch+1}/{deit_epochs} - Val Top1: {val['top1']:.2f}% | Best: {best_deit_acc:.2f}% ({epoch_duration:.1f}s)")
    else:
        print(f"Epoch {epoch+1}/{deit_epochs} finished in {epoch_duration:.1f}s (Eval skipped)")

print(f"\nDeiT Student training complete. Best model saved to {deit_save_path}")

# 6. Load Best Weights & Evaluate
deit_student.load_state_dict(torch.load(deit_save_path, map_location=DEVICE, weights_only=True))
deit_eval = evaluate(deit_student, test_loader, is_deit=True)

print("\nEvaluation Results for Best DeiT-Tiny Student:")
print(f"{'Model':<14} | {'top1':<6} | {'top5':<6} | {'macro-F1':<8} | {'many':<6} | {'medium':<6} | {'few':<6}")
print("-" * 65)
print(f"{'DeiT-Student':<14} | {deit_eval['top1']:>5.1f}% | {deit_eval['top5']:>5.1f}% | {deit_eval['macro_f1']:>8.3f} | {deit_eval['group_acc']['many']:>5.1f}% | {deit_eval['group_acc']['medium']:>5.1f}% | {deit_eval['group_acc']['few']:>4.1f}%")